# Bab 10: Pengenalan Jaringan Saraf Konvolusional (CNN)
*(Introduction to Convolutional Neural Networks)*

## Eksplorasi Teoritis Mendalam (Theoretical Deep-Dive)

Bab ini memecahkan salah satu masalah terbesar dalam *Deep Learning* modern: **Computer Vision** (membuat mesin memahami gambar). Jaringan saraf biasa (Fully Connected Networks) yang kita pelajari di bab-bab sebelumnya ternyata sangat buruk dalam memproses gambar. Mengapa? Karena mereka memperlakukan setiap piksel sebagai input yang benar-benar terpisah dan tidak memahami konsep "ruang" atau "kedekatan". Jaringan Saraf Konvolusional (CNN) diciptakan untuk meniru cara kerja korteks visual manusia.

### 1. Masalah pada Jaringan Saraf Biasa untuk Gambar
Jika kita memiliki gambar wajah berukuran $1000 \times 1000$ piksel, itu berarti ada 1 juta piksel input. Jika *hidden layer* pertama memiliki 1.000 node, maka model membutuhkan **1 miliar bobot (weights)** hanya untuk lapisan pertama!
*   **Inefisiensi:** Parameter yang terlalu banyak membuat komputer kehabisan memori dan pelatihannya memakan waktu berminggu-minggu.
*   **Posisi Kaku:** Jika model biasa diajarkan bahwa "hidung" ada di tengah gambar, lalu kita memberinya gambar wajah yang agak bergeser ke kiri, model tidak akan mengenalinya. Ia tidak memiliki sifat *Translation Invariance* (kemampuan mengenali objek terlepas dari posisinya).

### 2. Solusi: Menggunakan Kembali Bobot (Weight Sharing)
Ide brilian dari CNN adalah menggunakan kembali sekumpulan bobot yang sama di seluruh bagian gambar.
Daripada memiliki 1 miliar bobot spesifik untuk setiap lokasi piksel, kita hanya membuat sebuah "Filter" kecil (misalnya matriks bobot berukuran $3 \times 3$). Filter ini bertugas mendeteksi satu pola spesifik (misalnya garis horizontal, sudut, atau lengkungan). Kita menggeser filter yang sama ini ke seluruh permukaan gambar.

### 3. Operasi Konvolusi (The Convolution)
Secara matematis, apa itu Konvolusi? Ini hanyalah *Dot Product* (seperti yang kita pelajari di Bab 3), tetapi diterapkan secara berulang kali.
*   **Mekanisme Jendela Geser (Sliding Window):** Filter $3 \times 3$ diletakkan di sudut kiri atas gambar, dikalikan dengan piksel-piksel di bawahnya, dan dijumlahkan menjadi 1 angka baru. Kemudian filter digeser 1 langkah ke kanan, dikalikan lagi, dijumlahkan lagi. Proses ini diulang hingga seluruh gambar tersapu bersih.
*   **Peta Fitur (Feature Map):** Hasil dari sapuan filter ini adalah gambar baru (yang ukurannya sedikit lebih kecil). Jika filter tersebut bertugas mencari "tepi vertikal", maka gambar baru yang dihasilkan akan menyala terang (angka tinggi) di area mana pun yang memiliki tepi vertikal, dan gelap (angka 0) di area yang datar.

### 4. Hierarki Fitur Visual
Sama seperti jaringan mendalam biasa, lapisan konvolusi bisa ditumpuk. Lapisan konvolusi pertama mendeteksi garis dan sudut sederhana. Lapisan kedua menyatukan sudut-sudut tersebut untuk mendeteksi bentuk (seperti lingkaran mata). Lapisan yang lebih dalam lagi menyatukan bentuk-bentuk itu untuk mendeteksi wajah secara utuh.

In [ ]:
import numpy as np

# =====================================================================
# REPRODUKSI KODE BAB 10: JARINGAN SARAF KONVOLUSIONAL (CNN)
# Mendemonstrasikan operasi "Sliding Window" (Jendela Geser) Konvolusi
# =====================================================================

np.random.seed(1)

# 1. Membuat Gambar Tiruan (6x6 pixels)
# Mari kita asumsikan 0 adalah latar belakang gelap,
# dan angka yang lebih tinggi adalah garis vertikal terang di tengah.
image = np.array([
    [0, 0, 8, 8, 0, 0],
    [0, 0, 8, 8, 0, 0],
    [0, 0, 8, 8, 0, 0],
    [0, 0, 8, 8, 0, 0],
    [0, 0, 8, 8, 0, 0],
    [0, 0, 8, 8, 0, 0]
])

# 2. Membuat Filter/Kernel Konvolusi (3x3)
# Filter ini di-hardcode untuk mendeteksi "Tepi Kiri" (Transisi dari gelap ke terang)
# Kolom kiri negatif, kolom kanan positif.
filter_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

print("Gambar Asli (Garis Vertikal di Tengah):")
print(image)
print("\nFilter Deteksi Tepi Kiri:")
print(filter_kernel)
print("-" * 50)

def convolution_2d(image, kernel):
    # Mengambil dimensi gambar dan kernel
    image_row, image_col = image.shape
    kernel_row, kernel_col = kernel.shape

    # Menghitung ukuran output gambar baru setelah dikonvolusi
    output_row = image_row - kernel_row + 1
    output_col = image_col - kernel_col + 1
    output = np.zeros((output_row, output_col))

    # =======================================
    # OPERASI JENDELA GESER (SLIDING WINDOW)
    # =======================================
    for r in range(output_row):
        for c in range(output_col):
            # Memotong bagian gambar seukuran kernel (Sliding Window)
            image_section = image[r : r + kernel_row, c : c + kernel_col]

            # Melakukan perkalian elemen dan menjumlahkannya (Dot Product Konvolusi)
            output[r, c] = np.sum(image_section * kernel)

    return output

# 3. Menjalankan Konvolusi
feature_map = convolution_2d(image, filter_kernel)

print("\nHasil Peta Fitur (Feature Map) Setelah Konvolusi:")
print(feature_map)
print("\nPerhatikan bahwa nilai '24' yang positif menunjukkan titik di mana tepi kiri terdeteksi!")

### Penjelasan Eksekusi Kode:

1.  **Representasi Gambar (`image`):** Gambar direpresentasikan sebagai matriks angka. Pola `[0, 0, 8, 8, 0, 0]` menggambarkan transisi visual yang jelas: piksel gelap (0), bertemu objek terang (8), lalu kembali gelap (0). Ini membentuk garis lurus tebal secara vertikal.
2.  **Filter / Kernel (`filter_kernel`):** Matriks berukuran $3 \times 3$ ini bertindak sebagai pendeteksi. Konfigurasi `[-1, 0, 1]` dirancang secara matematis untuk mencari area di mana angka melonjak dari kecil ke besar. Jika diletakkan di area datar (misalnya `0 0 0`), hasilnya `0`. Jika diletakkan di tepi di mana warna berubah dari gelap ke terang, hasilnya akan menjadi angka yang besar.
3.  **Proses `convolution_2d`:** Inilah jantung dari CNN. Perulangan `for r` dan `for c` bertugas menggeser filter dari kiri ke kanan, dan dari atas ke bawah. Potongan gambar (sebesar $3 \times 3$) diambil di `image_section`, lalu dikalikan secara langsung dengan `kernel`.
4.  **Hasil *Feature Map*:** Gambar asli berukuran $6 \times 6$ menyusut menjadi peta fitur berukuran $4 \times 4$. Nilai `24` pada kolom pertama peta fitur adalah "sinyal kuat" bahwa filter telah berhasil menemukan tepi kiri dari garis vertikal pada gambar asli. Filter yang sama, dengan bobot yang sama, digunakan untuk memeriksa *setiap inci* gambar tersebut (*Weight Sharing*).